In [32]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import SelectFromModel
from sklearn.svm import SVR
import numpy as np

In [2]:
X,y = load_diabetes(return_X_y=True, as_frame=True)

In [3]:
X.shape, y.mean()

((442, 10), 152.13348416289594)

In [4]:
for model in [LinearRegression(), SVR()]:
    print(f'Model: {model.__class__}')
    print(f"R2 Score : {np.mean(cross_val_score(model,X,y,cv=3,scoring='r2'))}")
    print('---------')

Model: <class 'sklearn.linear_model._base.LinearRegression'>
R2 Score : 0.4887021298035315
---------
Model: <class 'sklearn.svm._classes.SVR'>
R2 Score : 0.13934818371190674
---------


In [5]:
X.shape, y.shape

((442, 10), (442,))

In [36]:
def do_hyper_search(model,param_grid,type='grid'):
    print(type)
    if type=='grid':
        clf = GridSearchCV(model, param_grid, scoring='r2', cv=3, n_jobs=-1)
        clf.fit(X.values,y)
    if type=='random':
        clf = RandomizedSearchCV(model, param_grid, scoring='r2', cv=3, n_jobs=-1)
        clf.fit(X.values,y)
    print(f"Best estimator : {clf.best_estimator_}")
    print(f"Best score: {clf.best_score_}")
    return clf


In [37]:
model = SVR()
param_grid = {'kernel':('linear', 'poly', 'rbf', 'sigmoid'),
                'degree':[1,2,3,4,5],
                'C':[1,10,100]
                }
clf_1 = do_hyper_search(model, param_grid, 'grid')
print('-------------')
clf_2 = do_hyper_search(model, param_grid, 'random')    

grid
Best estimator : SVR(C=10, degree=1, kernel='sigmoid')
Best score: 0.48633872054968585
-------------
random
Best estimator : SVR(C=100, degree=1, kernel='poly')
Best score: 0.48321242443917817


In [69]:
selector = SelectFromModel(SVR(kernel='linear')
                           , threshold='median',).fit(X,y)

In [70]:
selector.transform(X).shape

(442, 5)

### StandardScalerClone Implementation from Scratch

In [134]:
from sklearn.base import TransformerMixin
from sklearn.utils.validation import check_array, check_is_fitted
import pandas as pd
import numpy as np

In [174]:
class StandardScalerClone(TransformerMixin):
    def __init__(self, with_mean=True):
        self.with_mean = with_mean

    def fit(self, X, y=None):
        if isinstance(X, pd.DataFrame):
            self.feature_names_in = np.array(X.columns)
        else:
            self.feature_names_in = None
        X = check_array(X)
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0)
        self.n_features_in_ = X.shape[1]
        return self
    
    def transform(self, X):
        check_is_fitted(self)
        X = check_array(X)
        assert X.shape[1] == self.n_features_in_
        if self.with_mean:
            X = X - self.mean_
        return X / self.std_
    
    def inverse_transform(self, X):
        return (X * self.std_) + self.mean_
    
    def get_features_name_out(self):
        if isinstance(self.feature_names_in, np.ndarray):
            return self.feature_names_in
        else:
            return ['x'+str(i) for i in range(self.n_features_in_)]

In [175]:
ss = StandardScalerClone()

In [176]:
X

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641
...,...,...,...,...,...,...,...,...,...,...
437,0.041708,0.050680,0.019662,0.059744,-0.005697,-0.002566,-0.028674,-0.002592,0.031193,0.007207
438,-0.005515,0.050680,-0.015906,-0.067642,0.049341,0.079165,-0.028674,0.034309,-0.018114,0.044485
439,0.041708,0.050680,-0.015906,0.017293,-0.037344,-0.013840,-0.024993,-0.011080,-0.046883,0.015491
440,-0.045472,-0.044642,0.039062,0.001215,0.016318,0.015283,-0.028674,0.026560,0.044529,-0.025930


In [177]:
pd.DataFrame(ss.inverse_transform(ss.fit_transform(X.values)))

,0,1,2,3,4,5,6,7,8,9
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641
...,...,...,...,...,...,...,...,...,...,...
437,0.041708,0.050680,0.019662,0.059744,-0.005697,-0.002566,-0.028674,-0.002592,0.031193,0.007207
438,-0.005515,0.050680,-0.015906,-0.067642,0.049341,0.079165,-0.028674,0.034309,-0.018114,0.044485
439,0.041708,0.050680,-0.015906,0.017293,-0.037344,-0.013840,-0.024993,-0.011080,-0.046883,0.015491
440,-0.045472,-0.044642,0.039062,0.001215,0.016318,0.015283,-0.028674,0.026560,0.044529,-0.025930


In [178]:
ss.get_features_name_out()

['x0', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9']